## Biblioteki

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

from sklearn.ensemble import VotingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## Generowanie Cech

In [2]:
def generate_features(train_logs):
    
    chars_per_essay = train_logs[train_logs['activity'] == 'Input'].groupby('id').agg(
        chars_per_essay=('event_id', 'nunique')
    )



    grouped_train_logs = train_logs.groupby('id').agg(
        words_per_essay=('word_count', 'last'),
        max_up_time=('up_time', 'max'),
        min_down_time=('down_time', 'min'),
        events_per_essay=('event_id', 'nunique')
    )

 

    grouped_train_logs['writing_duration'] = grouped_train_logs['max_up_time'] - grouped_train_logs['min_down_time']

    grouped_train_logs = grouped_train_logs.merge(chars_per_essay, left_index=True, right_index=True)

    grouped_train_logs['chars_per_minute'] = 1000 * 60 * grouped_train_logs['chars_per_essay'] / grouped_train_logs['writing_duration']

    ### Pauzy

    train_logs.sort_values(by=['id', 'down_time'], inplace=True)

    train_logs['pause_time'] = (train_logs.groupby('id')['down_time'].shift(0) - train_logs.groupby('id')['up_time'].shift(1)).fillna(0)

  

    train_logs['is_pause'] = train_logs['pause_time'] > 2000

    pause_features = train_logs.groupby('id').agg(
        total_pauses=('is_pause', 'sum'),
        total_pause_time=('pause_time', 'sum')
    )


    grouped_train_logs = grouped_train_logs.merge(pause_features, left_index=True, right_index=True)

  

    grouped_train_logs['perc_pause_time'] = grouped_train_logs['total_pause_time'] / grouped_train_logs['writing_duration']

    grouped_train_logs['pauses_per_words'] = grouped_train_logs['total_pauses'] / grouped_train_logs['words_per_essay']

    ### Aktywności

   

    train_logs['text_change_len'] = train_logs['text_change'].str.len()

  
    train_logs.loc[train_logs['text_change'] == 'NoChange', 'text_change_len'] = 0

  

    activity_per_essay = {}
    activites = ['Remove/Cut', 'Nonproduction', 'Replace', 'Paste']

    for act in activites:
        activity_per_essay[act] = train_logs[train_logs['activity'] == act].groupby('id').agg(
            activity_per_essay=('event_id', 'nunique'),
            activity_len_per_essay=('text_change_len', 'sum')

        )

    activity_per_essay['Remove/Cut']

    for act in activites:
        grouped_train_logs = grouped_train_logs.merge(activity_per_essay[act],
                                                      left_index=True,
                                                      right_index=True,
                                                      suffixes=('', f"_{act}"), how='left')



    grouped_train_logs.rename(columns={'activity_per_essay': 'removes_per_essay',
                                       'activity_len_per_essay': 'removes_len_per_essay'}, inplace=True)

    grouped_train_logs['removes_per_words'] = grouped_train_logs['removes_per_essay'] / grouped_train_logs['words_per_essay']

    grouped_train_logs['replaces_per_words'] = grouped_train_logs['activity_per_essay_Replace'] / grouped_train_logs['words_per_essay']

    grouped_train_logs.fillna(0, inplace=True)

   


    ## Zaawansowane cechy

    grouped_train_logs['chars_per_words'] = grouped_train_logs['chars_per_essay'] / grouped_train_logs['words_per_essay']

    grouped_train_logs.reset_index(inplace=True)



    train_logs = train_logs.merge(grouped_train_logs[['id', 'min_down_time', 'writing_duration', 'words_per_essay']], on='id')

    train_logs['perc_time'] = (train_logs['up_time'] - train_logs['min_down_time']) / train_logs['writing_duration']
    train_logs['perc_words'] = train_logs['word_count'] / train_logs['words_per_essay']


    thres_res = {}

    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

    for t in thresholds:
        subset = train_logs[train_logs['perc_words'] >= t]
        subset = pd.DataFrame(subset.groupby('id')['perc_time'].min())
        thres_res[t] = subset

    thres_tab = thres_res[thresholds[0]][['perc_time']]

    for t in thresholds[1:]:
        thres_tab = thres_tab.merge(thres_res[t][['perc_time']],
                                    left_index=True, right_index=True,
                                    suffixes=('', f"_{t}"))

    thres_cols = [f"perc_time_{t}" for t in thresholds]
    thres_tab.columns = thres_cols

 

    grouped_train_logs = grouped_train_logs.merge(thres_tab.reset_index(), on='id')


    ## Burst


    train_logs['group'] = (train_logs['is_pause'].cumsum().shift(fill_value=0))


    grouped = train_logs[~train_logs['is_pause'] & (train_logs['activity'] == 'Input')].groupby(['id', 'group'])

    writing_time_no_pause = grouped['up_time'].last() - grouped['down_time'].first()



    words_no_pause = grouped['word_count'].last() - grouped['word_count'].first()

  

    combined = pd.DataFrame({'writing_time_no_pause': writing_time_no_pause,
                             'words_no_pause': words_no_pause}).reset_index()

    train_logs = train_logs.merge(combined, on=['id', 'group'], how='left')

   

    burst_tab = train_logs.groupby('id')[['words_no_pause', 'writing_time_no_pause']].max().reset_index()

    n_bursts = train_logs.groupby('id')['group'].nunique()

    burst_tab = burst_tab.merge(n_bursts.reset_index())

  

    grouped_train_logs = grouped_train_logs.merge(burst_tab)

   

    grouped_train_logs['bursts_per_words'] = grouped_train_logs['group'] / grouped_train_logs['words_per_essay']
    grouped_train_logs['words_no_pause_per_words'] = grouped_train_logs['words_no_pause'] / grouped_train_logs['words_per_essay']
    grouped_train_logs['writin_time_no_pause_per_time'] = grouped_train_logs['writing_time_no_pause'] / grouped_train_logs['writing_duration']

    return grouped_train_logs

## Przygotowanie zbiorów

In [3]:
kaggle_path = Path("/kaggle/input/linking-writing-processes-to-writing-quality")
local_path = Path("/Users/patrykksiazek/Desktop/Machine-Learning---Preliminary-Data-Analysis")
data_path = kaggle_path if kaggle_path.exists() else local_path

df = pd.read_csv(data_path / "train_logs.csv")
test_df = pd.read_csv(data_path / "test_logs.csv")
df_scores = pd.read_csv(data_path / "train_scores.csv")
sample_submission = pd.read_csv(data_path / "sample_submission.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/linking-writing-processes-to-writing-quality/train_logs.csv'

In [4]:
df_features = generate_features(df)

NameError: name 'df' is not defined